# Gaussian Copula

First of the four generation notebooks. Each follows the same shape: load the shared
training data, fit the generator, produce a synthetic dataset of the same size, save it,
and record how long training and generation took. One method per notebook means each gets
identical treatment, and a problem with one cannot interrupt the others.

Gaussian Copula is the statistical baseline. It learns the shape of each column and the
correlations between them, then samples from those shapes in a way that respects the
correlations. No neural networks are involved, which makes it much the fastest of the four
and a useful reference point: if the more complex methods cannot beat it, their extra cost
is hard to justify.

Its known limitation, covered in the literature review, is that correlation only captures
straight-line relationships. More complicated interactions can be missed, and the
evaluation will show whether that matters for this cohort.

## Setup

Gaussian Copula comes from sdv, which also provides CTGAN and TVAE. No GPU is needed;
this method fits in seconds.

In [1]:
%pip install -q pandas pyarrow sdv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.9/209.9 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.4/15.4 MB 143.0 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.1/75.1 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.0/207.0 kB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 98.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 kB 10.6 MB/s eta 0:00:00


## Working folder

Sets the project folder so everything the pipeline writes, the cohort, the synthetic
datasets, the outputs and the figures, persists between sessions rather than sitting on
temporary storage.

The cohort and the synthetic datasets derive from MIMIC-IV, which is credentialed data
under a PhysioNet data use agreement. Keep the folder private, do not share it, and
delete the data once the work is finished.

In [2]:
import os
from pathlib import Path

# Use the shared project folder when one is available, otherwise stay in the
# current directory.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    project_dir = Path("/content/drive/MyDrive/mimic-synthetic-pipeline")
    project_dir.mkdir(parents=True, exist_ok=True)
    os.chdir(project_dir)
    print(f"Working folder: {project_dir}")
except ImportError:
    print("Using the local working folder.")

Mounted at /content/drive
Working folder set to Google Drive: /content/drive/MyDrive/mimic-synthetic-pipeline


## Training data

Only the training set is loaded. The generator must never see the test set, which is what
the evaluation later uses to judge whether synthetic-trained models generalise to unseen
patients.

In [3]:
from pathlib import Path

import pandas as pd

if not Path("data/train.parquet").exists():
    raise FileNotFoundError(
        "data/train.parquet not found. Run the extraction and preparation steps first."
    )

TARGET = "readmitted_30d"
train_df = pd.read_parquet("data/train.parquet")
print(f"Training data: {len(train_df):,} admissions, readmission rate {train_df[TARGET].mean():.4f}")

Training data: 427,408 admissions, readmission rate 0.2067


## Hyperparameter search

An earlier round of this project ran every method at library defaults, with the training
budget set by the convergence rule. This section establishes whether Gaussian Copula performs better under other
settings, so the comparison between methods is not merely a comparison of defaults.

### Method

The search uses successive halving. Every configuration is screened on a small sample at a
reduced budget, and only the strongest few are promoted to a larger sample, the full budget
and repeated seeds. Configurations that look unpromising are therefore abandoned early,
which is where the saving in computation comes from, and the survivors are assessed with
enough repetition that the choice between them is not made on a single noisy run.

Neither library used in this project supports resuming training or validation-based early
stopping at a practical cost, so stopping is applied at the level of the configuration
rather than the epoch. The convergence rule used elsewhere in this project is applied to
each trial's loss curve and reported alongside its score, so a configuration that had not
finished training is visible rather than silently accepted.

Selection uses utility measured on the validation split. The test set is never involved, so
no part of the search can influence the reported results. Fidelity is recorded for every
trial but is not optimised, which means any movement in it is a consequence of selecting for
utility rather than a target of the search. That relationship is itself a finding worth
reporting.

Configurations whose mean scores fall within 0.005 of the best are reported as
indistinguishable, following the same reasoning applied to the differences between methods.
Where several are tied, the simplest should be preferred.

### Parameters varied

Both parameters the synthesizer exposes. `default_distribution` forces a single marginal family across every column, and `numerical_distributions` assigns them per column. The distinction matters here because length of stay is heavily right-skewed while age is bounded and capped at 91, so a single family is unlikely to suit both.

### Cost and outputs

Nine configurations. The copula fits in seconds, so the whole search completes in a few minutes.

Three files are written: every individual trial, a summary averaged over seeds, and the
selected configuration as JSON. The training cell below reads the selected
configuration automatically and applies it to the full training split. Nothing here overwrites the saved
synthetic datasets.

In [ ]:
import json
import time

import numpy as np
import pandas as pd
import torch
from pathlib import Path
from scipy.stats import ks_2samp
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# --- Search budget. Reduce these if the search needs to finish sooner. ----------------
SCREEN_ROWS = 25_000     # rows per trial in the screening round
SCREEN_FRACTION = 0.33   # fraction of the full training budget used when screening
PROMOTE_ROWS = 100_000   # rows per trial once a configuration is promoted
PROMOTE_KEEP = 3         # configurations carried into the promotion round
PROMOTE_SEEDS = (0, 1, 2)  # repeats per promoted configuration
TIE_THRESHOLD = 0.005    # ROC AUC difference treated as indistinguishable
FIDELITY_TOLERANCE = 1.5 # a candidate may not worsen KS or TVD beyond this multiple of
                         # the library default, however much utility it gains

NUMERIC_COLS = ["age_at_admission", "length_of_stay_days"]
CATEGORICAL_COLS = [
    "gender", "admission_type", "admission_location", "insurance",
    "marital_status", "race", "language", "had_icu_stay",
]

for _required in ["data/train_fit.parquet", "data/train_val.parquet"]:
    if not Path(_required).exists():
        raise FileNotFoundError(
            f"{_required} not found. Re-run 02_data_preparation.ipynb, which writes the "
            "validation split this search depends on."
        )

fit_df = pd.read_parquet("data/train_fit.parquet")
val_df = pd.read_parquet("data/train_val.parquet")
for _c in fit_df.columns:
    if str(fit_df[_c].dtype) == "Int64":
        fit_df[_c] = fit_df[_c].astype("int64")
        val_df[_c] = val_df[_c].astype("int64")

out_dir = Path("outputs")
out_dir.mkdir(exist_ok=True)

print(f"Fitting split {len(fit_df):,} rows | validation split {len(val_df):,} rows")


def make_classifier() -> Pipeline:
    """The classifier from the main evaluation, so scores are directly comparable."""
    preprocess = ColumnTransformer([
        ("num", StandardScaler(), NUMERIC_COLS),
        ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL_COLS),
    ])
    return Pipeline([
        ("preprocess", preprocess),
        ("clf", LogisticRegression(max_iter=1000, class_weight="balanced")),
    ])


def utility_on_validation(synthetic: pd.DataFrame) -> float:
    """Train on synthetic, score on the real validation split. The test set is never used."""
    clf = make_classifier()
    clf.fit(synthetic.drop(columns=[TARGET]), synthetic[TARGET])
    proba = clf.predict_proba(val_df.drop(columns=[TARGET]))[:, 1]
    return float(roc_auc_score(val_df[TARGET], proba))


def _tvd(real_col: pd.Series, syn_col: pd.Series) -> float:
    p = real_col.value_counts(normalize=True)
    q = syn_col.value_counts(normalize=True)
    return 0.5 * sum(abs(p.get(c, 0.0) - q.get(c, 0.0)) for c in p.index.union(q.index))


def fidelity_summary(synthetic: pd.DataFrame, real: pd.DataFrame) -> tuple:
    ks = float(np.mean([
        ks_2samp(real[c].astype(float), synthetic[c].astype(float)).statistic
        for c in NUMERIC_COLS
    ]))
    tvd = float(np.mean([
        _tvd(real[c].astype(str), synthetic[c].astype(str)) for c in CATEGORICAL_COLS
    ]))
    return ks, tvd


def converged(loss_series) -> tuple:
    """The stopping rule used elsewhere: mean loss over the final fifth of training
    against the fifth before it. Returns the percentage improvement and a flag."""
    if loss_series is None or len(loss_series) < 10:
        return float("nan"), None
    values = np.asarray(loss_series, dtype=float)
    n = len(values)
    previous = values[int(n * 0.6):int(n * 0.8)].mean()
    final = values[int(n * 0.8):].mean()
    improvement = (previous - final) / abs(previous) * 100
    return float(improvement), bool(improvement <= 1.0)


def set_seed(seed: int) -> None:
    """The sdv synthesizers expose no seed argument, so the global generators are set."""
    np.random.seed(seed)
    torch.manual_seed(seed)


def run_search(space, fit_and_sample, method_slug, full_budget):
    """Successive halving. Every configuration is screened on a small sample at a reduced
    budget; the best few are promoted to a larger sample, a full budget and repeated seeds.
    Unpromising configurations are therefore stopped early, which is where the compute
    saving comes from."""
    screen_sample = fit_df.sample(n=min(SCREEN_ROWS, len(fit_df)), random_state=0).reset_index(drop=True)
    screen_budget = max(1, int(full_budget * SCREEN_FRACTION))

    print(f"\n{'=' * 78}")
    print(f"ROUND 1, screening {len(space)} configurations")
    print(f"{len(screen_sample):,} rows, budget {screen_budget}, one seed each")
    print(f"{'=' * 78}", flush=True)

    screened = []
    for i, (label, config) in enumerate(space, start=1):
        print(f"\n[{i}/{len(space)}] {label}", flush=True)
        t0 = time.time()
        try:
            set_seed(0)
            synthetic, losses = fit_and_sample(config, screen_sample, screen_budget)
            auc = utility_on_validation(synthetic)
            ks, tvd = fidelity_summary(synthetic, screen_sample)
            improvement, is_converged = converged(losses)
            seconds = time.time() - t0
            screened.append({
                "config_label": label, "config": json.dumps(config), "round": "screen",
                "val_roc_auc": auc, "mean_ks": ks, "mean_tvd": tvd,
                "loss_improvement_pct": improvement, "converged": is_converged,
                "seconds": seconds,
            })
            flag = "" if is_converged is None else ("converged" if is_converged else "NOT converged")
            print(f"      ROC AUC {auc:.4f} | KS {ks:.4f} | TVD {tvd:.4f} | {seconds:.0f}s {flag}", flush=True)
        except Exception as exc:  # noqa: BLE001
            print(f"      failed: {type(exc).__name__}: {exc}", flush=True)

    if not screened:
        raise RuntimeError("Every configuration failed during screening.")

    screen_df = pd.DataFrame(screened).sort_values("val_roc_auc", ascending=False)

    # Utility alone is not a sufficient selection criterion. A configuration can raise the
    # downstream score while badly degrading the distributions, which would be a poor
    # outcome for a project whose argument is that these dimensions must be read together.
    # Candidates are therefore restricted to those whose fidelity is no worse than the
    # first configuration in the search space, the library default, by more than the
    # tolerance below. Utility decides the ranking within that set.
    baseline_label = space[0][0]
    baseline = screen_df[screen_df["config_label"] == baseline_label]
    eligible = screen_df
    if not baseline.empty:
        base_ks = float(baseline.iloc[0]["mean_ks"])
        base_tvd = float(baseline.iloc[0]["mean_tvd"])
        limit_ks = base_ks * FIDELITY_TOLERANCE + 1e-6
        limit_tvd = base_tvd * FIDELITY_TOLERANCE + 1e-6
        eligible = screen_df[(screen_df["mean_ks"] <= limit_ks)
                             & (screen_df["mean_tvd"] <= limit_tvd)]
        excluded = screen_df[~screen_df["config_label"].isin(eligible["config_label"])]
        print()
        print(f"Fidelity guardrail: KS <= {limit_ks:.4f}, TVD <= {limit_tvd:.4f}")
        print(f"  ({FIDELITY_TOLERANCE}x the baseline configuration '{baseline_label}')")
        if len(excluded):
            print(f"  Excluded {len(excluded)} configuration(s) that raised utility at the "
                  f"cost of fidelity:")
            for _, r in excluded.iterrows():
                print(f"    {r['config_label']:<42} ROC AUC {r['val_roc_auc']:.4f}  "
                      f"KS {r['mean_ks']:.4f}  TVD {r['mean_tvd']:.4f}")
        if eligible.empty:
            print("  No configuration met the guardrail, so it has been relaxed for this run.")
            eligible = screen_df

    promoted = eligible.head(PROMOTE_KEEP)

    print(f"\n{'=' * 78}")
    print(f"ROUND 2, promoting the top {len(promoted)} of {len(screen_df)}")
    print(f"{min(PROMOTE_ROWS, len(fit_df)):,} rows, budget {full_budget}, "
          f"{len(PROMOTE_SEEDS)} seeds each")
    print(f"{'=' * 78}", flush=True)

    promote_sample = fit_df.sample(n=min(PROMOTE_ROWS, len(fit_df)), random_state=0).reset_index(drop=True)
    promoted_rows = []
    for i, row in enumerate(promoted.itertuples(index=False), start=1):
        config = json.loads(row.config)
        print(f"\n[{i}/{len(promoted)}] {row.config_label}", flush=True)
        for seed in PROMOTE_SEEDS:
            t0 = time.time()
            try:
                set_seed(seed)
                synthetic, losses = fit_and_sample(config, promote_sample, full_budget, seed=seed)
                auc = utility_on_validation(synthetic)
                ks, tvd = fidelity_summary(synthetic, promote_sample)
                improvement, is_converged = converged(losses)
                promoted_rows.append({
                    "config_label": row.config_label, "config": row.config, "round": "promote",
                    "seed": seed, "val_roc_auc": auc, "mean_ks": ks, "mean_tvd": tvd,
                    "loss_improvement_pct": improvement, "converged": is_converged,
                    "seconds": time.time() - t0,
                })
                print(f"      seed {seed}: ROC AUC {auc:.4f} | KS {ks:.4f} | TVD {tvd:.4f}", flush=True)
            except Exception as exc:  # noqa: BLE001
                print(f"      seed {seed} failed: {type(exc).__name__}: {exc}", flush=True)

    promote_df = pd.DataFrame(promoted_rows)
    summary = (
        promote_df.groupby(["config_label", "config"])
        .agg(mean_roc_auc=("val_roc_auc", "mean"), sd_roc_auc=("val_roc_auc", "std"),
             mean_ks=("mean_ks", "mean"), mean_tvd=("mean_tvd", "mean"),
             runs=("val_roc_auc", "size"))
        .reset_index().sort_values("mean_roc_auc", ascending=False)
    )

    pd.concat([screen_df, promote_df], ignore_index=True).to_csv(
        out_dir / f"tuning_{method_slug}_trials.csv", index=False)
    summary.to_csv(out_dir / f"tuning_{method_slug}_summary.csv", index=False)

    print(f"\n{'=' * 78}")
    print("PROMOTION RESULTS, mean over seeds")
    print(f"{'=' * 78}")
    for _, r in summary.iterrows():
        sd = 0.0 if pd.isna(r["sd_roc_auc"]) else r["sd_roc_auc"]
        print(f"  {r['config_label']:<46} {r['mean_roc_auc']:.4f} +/- {sd:.4f}  "
              f"KS {r['mean_ks']:.4f}  TVD {r['mean_tvd']:.4f}  (n={r['runs']})")

    best = summary.iloc[0]
    tied = summary[summary["mean_roc_auc"] >= best["mean_roc_auc"] - TIE_THRESHOLD]

    checkpoint = {
        "method": method_slug,
        "selected_config": json.loads(best["config"]),
        "selected_label": best["config_label"],
        "mean_val_roc_auc": float(best["mean_roc_auc"]),
        "sd_val_roc_auc": None if pd.isna(best["sd_roc_auc"]) else float(best["sd_roc_auc"]),
        "seeds": list(PROMOTE_SEEDS),
        "screen_rows": int(min(SCREEN_ROWS, len(fit_df))),
        "promote_rows": int(min(PROMOTE_ROWS, len(fit_df))),
        "full_budget": full_budget,
        "tied_within_threshold": tied["config_label"].tolist(),
    }
    with open(out_dir / f"tuning_{method_slug}_selected.json", "w", encoding="utf-8") as fh:
        json.dump(checkpoint, fh, indent=2)

    print(f"\nSelected: {best['config_label']}")
    if len(tied) > 1:
        print(f"Within {TIE_THRESHOLD} of the best, so not separable on this evidence:")
        for label in tied["config_label"]:
            print(f"    {label}")
        print("Prefer the simplest of these, and say so in the write-up.")
    print(f"\nSaved outputs/tuning_{method_slug}_selected.json for the final run.")
    return summary


METHOD_SLUG = "gaussian_copula"
FULL_BUDGET = 1  # the copula has no iterative budget; kept for interface consistency

from sdv.metadata import Metadata
from sdv.single_table import GaussianCopulaSynthesizer

# Both levers the synthesizer exposes. default_distribution forces one marginal family
# across every column; numerical_distributions sets them per column, which matters here
# because length of stay is heavily right-skewed while age is bounded and capped at 91.
SEARCH_SPACE = [
    ("library defaults", {}),
    ("default=norm", {"default_distribution": "norm"}),
    ("default=beta", {"default_distribution": "beta"}),
    ("default=truncnorm", {"default_distribution": "truncnorm"}),
    ("default=gamma", {"default_distribution": "gamma"}),
    ("default=uniform", {"default_distribution": "uniform"}),
    ("per-column: gamma LoS, truncnorm age", {"numerical_distributions": {
        "length_of_stay_days": "gamma", "age_at_admission": "truncnorm"}}),
    ("per-column: beta LoS, norm age", {"numerical_distributions": {
        "length_of_stay_days": "beta", "age_at_admission": "norm"}}),
    ("per-column: gamma LoS, beta age", {"numerical_distributions": {
        "length_of_stay_days": "gamma", "age_at_admission": "beta"}}),
]


def fit_and_sample(config, data, budget, seed=0):
    metadata = Metadata.detect_from_dataframe(data, table_name="cohort")
    model = GaussianCopulaSynthesizer(metadata, **config)
    model.fit(data)
    return model.sample(num_rows=len(data)), None

summary = run_search(SEARCH_SPACE, fit_and_sample, METHOD_SLUG, FULL_BUDGET)
summary


## Fitting and generating

The model is fitted on the training data, then asked for a synthetic dataset with the
same number of rows. Matching the size keeps the comparison simple: every method produces
one synthetic training set of identical size.

Training and generation are timed and appended to a shared log. The four methods differ
sharply in computational cost, and the timings turn that into a measured result rather
than an assertion.

In [4]:
import time

from sdv.metadata import Metadata
from sdv.single_table import GaussianCopulaSynthesizer

import json as _json

# Set to False to reproduce the library-default run rather than the tuned one.
USE_TUNED_CONFIG = True

TUNED = {}
CONFIG_LABEL = "library defaults"
_selected = Path("outputs/tuning_gaussian_copula_selected.json")
if USE_TUNED_CONFIG and _selected.exists():
    with open(_selected, encoding="utf-8") as _fh:
        _payload = _json.load(_fh)
    TUNED = _payload["selected_config"]
    CONFIG_LABEL = _payload["selected_label"]
    print(f"Using the configuration chosen by the search: {CONFIG_LABEL}")
    print(f"  {TUNED}")
elif USE_TUNED_CONFIG:
    print("No search result found, so the library defaults are used.")
    print(f"  Run the search section at the end of this notebook to produce {_selected}.")
else:
    print("USE_TUNED_CONFIG is False, so the library defaults are used.")

metadata = Metadata.detect_from_dataframe(train_df, table_name="cohort")
model = GaussianCopulaSynthesizer(metadata, **TUNED)

t0 = time.time()
model.fit(train_df)
train_seconds = time.time() - t0

t0 = time.time()
synthetic_df = model.sample(num_rows=len(train_df))
generate_seconds = time.time() - t0

synthetic_df.to_parquet("data/synthetic_gaussian_copula.parquet", index=False)
print(f"Training took {train_seconds:.1f}s, generation took {generate_seconds:.1f}s")
print(f"Saved {len(synthetic_df):,} synthetic admissions to data/synthetic_gaussian_copula.parquet")

/usr/local/lib/python3.12/dist-packages/sdv/single_table/base.py:139: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(


Training took 105.5s, generation took 7.3s
Saved 427,408 synthetic admissions to data/synthetic_gaussian_copula.parquet


## Sanity checks

Not the full evaluation, just quick aggregate checks that the generator produced
something sensible. The readmission rate should sit near the real training rate of about
0.21, and the age and length-of-stay averages should be plausible. Catching an obviously
broken run here saves a wasted evaluation later.

In [5]:
checks = pd.DataFrame({
    "statistic": ["Readmission rate", "Mean age", "Mean length of stay (days)"],
    "real_training_data": [
        round(train_df[TARGET].mean(), 4),
        round(train_df["age_at_admission"].astype(float).mean(), 1),
        round(train_df["length_of_stay_days"].mean(), 2),
    ],
    "synthetic_data": [
        round(synthetic_df[TARGET].mean(), 4),
        round(synthetic_df["age_at_admission"].astype(float).mean(), 1),
        round(synthetic_df["length_of_stay_days"].mean(), 2),
    ],
})
checks

,statistic,real_training_data,synthetic_data
0,Readmission rate,0.2067,0.2088
1,Mean age,58.8000,59.6000
2,Mean length of stay (days),4.6400,4.2800


In [6]:
# Record the hardware alongside the timings. Without it the cost comparison in the
# results chapter rests on recollection of which session used which accelerator.
try:
    import torch as _torch
    GPU_NAME = _torch.cuda.get_device_name(0) if _torch.cuda.is_available() else "CPU"
except Exception:  # noqa: BLE001
    GPU_NAME = "unknown"

# Append this run's timings to the shared generation log used by all four methods.
out_dir = Path("outputs")
out_dir.mkdir(exist_ok=True)
log_path = out_dir / "generation_log.csv"

entry = pd.DataFrame([{
    "method": "Gaussian Copula",
    "rows_generated": len(synthetic_df),
    "train_seconds": round(train_seconds, 1),
    "generate_seconds": round(generate_seconds, 1),
    "gpu": GPU_NAME,
    "config": CONFIG_LABEL,
}])
if log_path.exists():
    log = pd.read_csv(log_path)
    # Keyed on method and configuration, so the default and tuned runs both survive.
    if "config" in log.columns:
        log = log[~((log["method"] == "Gaussian Copula") & (log["config"] == CONFIG_LABEL))]
    else:
        log = log[log["method"] != "Gaussian Copula"]
    log = pd.concat([log, entry], ignore_index=True)
else:
    log = entry
log.to_csv(log_path, index=False)
log

,method,rows_generated,train_seconds,generate_seconds
0,TVAE,427408,453.1,3.1
1,TabDDPM,427408,6153.8,83.4
2,CTGAN,427408,6724.9,5.5
3,Gaussian Copula,427408,105.5,7.3
